In [1]:
import pandas as pd
import re
import random

In [2]:
head_labels = 'Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime'.split(',')
columnNameToLabel = {}
for i in range(len(head_labels)):
    columnNameToLabel['column' + str(i+1)] = head_labels[i]

In [3]:
def lineToDataStruct(lineElements):

    index_dependant = 0
    for element in lineElements:
        if element.startswith('"dependant":'):
            break
        index_dependant += 1

    rhs = lineElements[index_dependant + 1].split(':')[1]
    for removeChar in ['"','[',']','{','}']:
        rhs = rhs.replace(removeChar, '')
    determinant_index = 0
    rhs = columnNameToLabel[rhs]

    for element in lineElements:
        if element.startswith('"determinant'):
            #print('found at ' + str(determinant_index))
            break
        determinant_index += 1

    X = [] # RHS
    for s in lineElements[determinant_index + 1: index_dependant]:
        match = re.search(r'"columnIdentifier":"(column\d+)"', s)
        if match:
            result = match.group(1)
            X.append(columnNameToLabel[result])

    return X,rhs

In [4]:
data = []
with open('2025-11-14_17-00-10_fds.csv', 'r') as file:
    for line in file:
        lhs, rhs = lineToDataStruct(line.strip().split(","))
        prettyPrint = lambda lst: ', '.join(lst)
        #print(prettyPrint(lhs), '->',rhs)
        data .append( (lhs, rhs) )

In [5]:
count = 0
for tuple in data:
    if all(str != 'WheelsOn' for str in tuple[0]):
        count+=1

In [6]:
ids = random.sample(range(len(data)), 100)

In [7]:
sample = [data[i] for i in ids]

In [8]:
sample[2]

(['OriginStateFips', 'DestAirportSeqID', 'ActualElapsedTime', 'NASDelay'],
 'SecurityDelay')

In [7]:
def find_fds_for_rhs(target_rhs, data,sorted=False, lhs_validator = lambda x: True):
    results = []
    for tuple in data:
        lhs, rhs = tuple
        if rhs == target_rhs and lhs_validator(lhs):
            results.append(tuple)
    if sorted:
        results.sort(key=lambda x: len(x[0]))
    return results

In [8]:
genuine_fds = [find_fds_for_rhs('ArrDelay', data, sorted = True)[0]]

In [9]:
firsts = [find_fds_for_rhs(label, data, sorted = True)[0] for label in head_labels]

In [10]:
firsts

[(['ArrTime'], 'Year'),
 (['ArrTime'], 'Quarter'),
 (['ArrTime'], 'Month'),
 (['FlightDate'], 'DayofMonth'),
 (['DayofMonth'], 'DayOfWeek'),
 (['DayofMonth'], 'FlightDate'),
 (['ArrTime'], 'UniqueCarrier'),
 (['ArrTime'], 'AirlineID'),
 (['ArrTime'], 'Carrier'),
 (['FlightNum', 'CRSArrTime', 'DayofMonth'], 'TailNum'),
 (['CRSDepTime', 'CRSArrTime', 'DayofMonth'], 'FlightNum'),
 (['OriginAirportSeqID'], 'OriginAirportID'),
 (['OriginAirportID'], 'OriginAirportSeqID'),
 (['OriginAirportID'], 'OriginCityMarketID'),
 (['OriginAirportID'], 'Origin'),
 (['OriginAirportID'], 'OriginCityName'),
 (['OriginAirportID'], 'OriginState'),
 (['OriginAirportID'], 'OriginStateFips'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportID'], 'OriginWac'),
 (['DestAirportSeqID'], 'DestAirportID'),
 (['DestAirportID'], 'DestAirportSeqID'),
 (['DestAirportID'], 'DestCityMarketID'),
 (['DestAirportID'], 'Dest'),
 (['DestAirportID'], 'DestCityName'),
 (['DestAirportID'], 'DestState'),
 (['DestAirport

In [11]:
def create_sample(data, n):
    ids = random.sample(range(len(data)), n)
    sample = [data[i] for i in ids]
    return sample

In [12]:
import csv
import os

def export_sample_to_csv(filename):
    if not os.path.exists(filename):
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            write_back = [['LHS', 'RHS']]
            write_back.extend(create_sample(data, 1000))
            writer.writerows(write_back)


In [13]:
export_sample_to_csv( 'training_data.csv')

In [14]:
find_fds_for_rhs('NASDelay', data, sorted = True)

[(['TailNum', 'DepTime', 'ArrTime'], 'NASDelay'),
 (['DepTime', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'ArrDelay'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'ArrDelayMinutes'], 'NASDelay'),
 (['DepTime', 'ArrTime', 'CarrierDelay'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'ActualElapsedTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'AirTime'], 'NASDelay'),
 (['TailNum', 'DepDelay', 'ArrTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'CarrierDelay'], 'NASDelay'),
 (['TailNum', 'DayofMonth', 'ArrTime'], 'NASDelay'),
 (['TailNum', 'ArrTime', 'FlightDate'], 'NASDelay'),
 (['CRSArrTime', 'ArrTime', 'ActualElapsedTime'], 'NASDelay'),
 (['DepDelay', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['DepDelayMinutes', 'CRSArrTime', 'ArrTime'], 'NASDelay'),
 (['CRSArrTime', 'DayofMonth', 'ArrTime'], 'NASDelay'),
 (['CRSArrTime', 'ArrTime', 'FlightDate'], 'NASDelay'),
 (['ArrTime', 'ArrDelay', 'ActualElapsedTime'], 'NASDelay'),
 (['ArrTime', 'ArrDelayMinutes', 'ActualElapsedTime'], 'NASD

In [15]:
find_fds_for_rhs('FlightNum', data, sorted = True, lhs_validator = lambda lhs: any(['origin' in label.lower() for label in lhs]) and not any(['city' in label.lower() for label in lhs]))

[(['TailNum', 'OriginAirportID', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'Origin', 'CRSDepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSArrTime', 'CRSElapsedTime'],
  'FlightNum'),
 (['TailNum', 'Origin', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginAirportSeqID', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'Origin', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginState', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginStateFips', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginStateName', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginWac', 'CRSArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'OriginAirportID', 'DestAi

In [16]:
def build_lhs_validator(required_substrings, forbidden_substrings):
    def lhs_validator(lhs):
        lhs_lower = [label.lower() for label in lhs]
        for req in required_substrings:
            if not any([req in label for label in lhs_lower]):
                return False
        for forb in forbidden_substrings:
            if any([forb in label for label in lhs_lower]):
                return False
        return True
    return lhs_validator

In [24]:
print(build_lhs_validator(['origin'], ['time'])(['city_name', 'origin_state']))

True


In [ ]:
find_fds_for_rhs('', data, sorted = True, lhs_validator = build_lhs_validator([], []))

[(['CRSDepTime', 'CRSArrTime', 'DayofMonth'], 'FlightNum'),
 (['CRSDepTime', 'CRSArrTime', 'FlightDate'], 'FlightNum'),
 (['TailNum', 'CRSDepTime', 'CRSArrTime', 'ArrTime'], 'FlightNum'),
 (['TailNum', 'CRSArrTime', 'ArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'CRSArrTime', 'ArrTime', 'Distance'], 'FlightNum'),
 (['TailNum', 'CRSDepTime', 'ArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['CRSArrTime', 'ArrTime', 'CRSElapsedTime', 'AirTime'], 'FlightNum'),
 (['CRSArrTime', 'DayofMonth', 'ArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['CRSArrTime', 'ArrTime', 'CRSElapsedTime', 'FlightDate'], 'FlightNum'),
 (['CRSDepTime', 'ArrTime', 'CRSElapsedTime', 'AirTime'], 'FlightNum'),
 (['CRSDepTime', 'DayofMonth', 'ArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['CRSDepTime', 'ArrTime', 'CRSElapsedTime', 'FlightDate'], 'FlightNum'),
 (['TailNum', 'CRSDepTime', 'DepTime', 'CRSArrTime'], 'FlightNum'),
 (['TailNum', 'DepTime', 'CRSArrTime', 'CRSElapsedTime'], 'FlightNum'),
 (['TailNum', 'Dep